# Agentic Eval Scorer Review

3 specialized reviewer agents with File Search access analyze eval scorer discrepancies and generate improved prompts:
1. **Reference Reviewer** — general_law + specific_law File Search
2. **Judgement Reviewer** — supreme_court File Search
3. **Suggestion Reviewer** — review only (no File Search)

## Setup

In [ ]:
from legal_rag.eval_review import EvalReviewClient, ScorerResult
from legal_rag.config import (
    GENERAL_LAW_FILE_STORE,
    SPECIFIC_LAW_FILE_STORE,
    SUPREME_COURT_STATEMENT_FILE_STORE,
)
from skill import load_prompt
from config import *

import json
import copy
import math
import os
import time
import pandas as pd

In [ ]:
from google import genai
from google.genai import types

client = genai.Client(api_key=os.getenv("GENAI_API_KEY"))

In [ ]:
EVAL_RESULTS_FOLDER = os.path.join(EVAL_RESULTS_ROOT_PATH, f"eval_scorer_{EVAL_AI_PROMPT_VERSION}")
EVAL_RESULTS_FOLDER

## 1. Initialize Review Client

In [ ]:
REVIEW_AI_PROMPT_VERSION = "v07"
NEW_EVAL_AI_PROMPT_VERSION = "v08"

reviewer = EvalReviewClient(
    eval_model=EVAL_AI_MODEL,
    review_model=REVIEW_AI_MODEL,
    eval_prompt_version=REVIEW_AI_PROMPT_VERSION,
    prompt_version=REVIEW_AI_PROMPT_VERSION,
)
reviewer

## 2. Load & Transform Eval Results

In [ ]:
data = EvalReviewClient.load_eval_folder(EVAL_RESULTS_FOLDER)

print(f"\nLoaded {len(data)} total entries\n")
for i, d in enumerate(data):
    print(f"Entry {i+1} [{d['source_file']}]: {d['input'][:80]}...")
    print(f"  Auto:  {d['auto_scores']}")
    print(f"  Human: {d['human_scores']}")
    print("-------------------------------")

In [ ]:
# Quick score comparison table
rows = []
for i, d in enumerate(data):
    for key in ["reference", "judgement", "suggestion"]:
        auto = d["auto_scores"][key]
        human = d["human_scores"][key]
        if human is None: continue
        rows.append({
            "entry": i + 1,
            "source_file": d["source_file"],
            "scorer": key,
            "auto_score": auto,
            "human_score": human,
            "diff": auto - human if auto is not None and human is not None else None,
        })

score_df = pd.DataFrame(rows)
score_df

## 3. Re-run Scorers

In [ ]:
SCORER_NAMES = ["reference", "judgement", "suggestion"]
LIMIT = 10
NUM_RERUN_ROWS = math.floor(LIMIT / len(SCORER_NAMES))
NUM_RERUN_TIMES = math.ceil(len(data) / NUM_RERUN_ROWS)

reviews = []
for i in range(NUM_RERUN_TIMES):
    start_idx = i * NUM_RERUN_ROWS
    end_idx = min((i + 1) * NUM_RERUN_ROWS, len(data))
    print(f"\nRe-running scorers for entries {start_idx + 1} to {end_idx}...")
    reviews.extend(reviewer.rerun_scorers(data[start_idx:end_idx], SCORER_NAMES))

    print("Sleeping for 20 seconds to avoid rate limits...")
    time.sleep(20)
    print("------------------------------------------------")

reviews

In [ ]:
# Save re-run results
with open(f'docs/KKP/LNC/review_results_{REVIEW_AI_PROMPT_VERSION}.json', 'w', encoding='utf-8') as f:
    res_list = []
    for r in reviews:
        temp_dict = copy.deepcopy(r.__dict__)
        temp_res = []
        for score in temp_dict['scorer_results']:
            temp_res.append(score.__dict__)
        temp_dict['scorer_results'] = temp_res
        res_list.append(temp_dict)
    json.dump(res_list, f, indent=4, ensure_ascii=False)

In [ ]:
# Compare re-run scores vs human scores
total_ref_df = {}

for scorer_name in SCORER_NAMES:
    rerun_rows = []
    for review in reviews:
        sr = review.scorer_results[0]
        rerun_rows.append({
            "entry": review.index + 1,
            "source_file": review.source_file,
            "rerun_score": sr.score,
            "original_auto": review.auto_scores.get(scorer_name),
            "human_score": review.human_scores.get(scorer_name),
        })
    ref_df = pd.DataFrame(rerun_rows)
    ref_df['dif'] = ref_df['rerun_score'] - ref_df['human_score']
    total_ref_df[scorer_name] = ref_df

for scorer_name in SCORER_NAMES:
    print(f"{scorer_name}: {total_ref_df[scorer_name]['dif'].abs().sum()}")

## 4. Prepare Review Data per Scorer

In [ ]:
# Build per-scorer review data for agent input
per_scorer_data = {}

for scorer_name in SCORER_NAMES:
    entries = []
    for review in reviews:
        sr = next((s for s in review.scorer_results if s.scorer_name == scorer_name), None)
        entries.append({
            "input": review.input,
            "output": review.output,
            "expected": review.expected,
            "human_score": review.human_scores.get(scorer_name),
            "auto_score": review.auto_scores.get(scorer_name),
            "rerun_score": sr.score if sr else None,
            "human_feedback": review.human_feedback,
            "scorer_rationale": sr.rationale if sr else "",
        })
    per_scorer_data[scorer_name] = entries

print(f"Reference: {len(per_scorer_data['reference'])} entries")
print(f"Judgement: {len(per_scorer_data['judgement'])} entries")
print(f"Suggestion: {len(per_scorer_data['suggestion'])} entries")

In [ ]:
# Load current scorer prompts
reference_prompt = load_prompt("eval_scorer", EVAL_AI_MODEL, "legal_reference_scorer", EVAL_AI_PROMPT_VERSION)
judgement_prompt = load_prompt("eval_scorer", EVAL_AI_MODEL, "legal_judgement_scorer", EVAL_AI_PROMPT_VERSION)
suggestion_prompt = load_prompt("eval_scorer", EVAL_AI_MODEL, "legal_suggestion_scorer", EVAL_AI_PROMPT_VERSION)

print(f"Reference prompt: {len(reference_prompt)} chars")
print(f"Judgement prompt: {len(judgement_prompt)} chars")
print(f"Suggestion prompt: {len(suggestion_prompt)} chars")

## 5. Reviewer Agents

Each agent analyzes discrepancies for its section and generates an improved scorer prompt.
- **Reference Reviewer**: File Search access to general_law + specific_law stores
- **Judgement Reviewer**: File Search access to supreme_court store
- **Suggestion Reviewer**: No File Search (review only)

In [ ]:
AGENT_INSTRUCTION = """คุณคือผู้เชี่ยวชาญทางกฎหมายที่ทำหน้าที่ตรวจสอบและปรับปรุง prompt ของ eval scorer

## บทบาท
วิเคราะห์ความแตกต่างระหว่างคะแนนที่มนุษย์ให้ (human_score) กับคะแนนที่ scorer ให้ (rerun_score/auto_score)
จากนั้นปรับปรุง prompt ของ scorer ให้ให้คะแนนสอดคล้องกับมนุษย์มากขึ้น

## เกณฑ์การให้คะแนนของ {scorer_section}
{scoring_criteria}

## ข้อมูลที่ได้รับ
- review_data: ข้อมูลผลการ re-run scorer แต่ละ entry พร้อม human_score, auto_score, human_feedback, scorer_rationale
- current_prompt: prompt ปัจจุบันของ scorer ที่ต้องปรับปรุง

## หน้าที่
1. **วิเคราะห์ Discrepancies**: สรุปแนวโน้มและ pattern ของความแตกต่างระหว่าง human vs scorer โดย{verification_instruction}
2. **ระบุจุดอ่อนของ prompt**: บอกว่า prompt ปัจจุบันมีจุดอ่อนตรงไหนที่ทำให้ scorer ให้คะแนนไม่ตรงกับมนุษย์
3. **สร้าง prompt ใหม่**: เขียน prompt ใหม่ทั้งหมดที่ปรับปรุงจุดอ่อนที่พบ

## รูปแบบผลลัพธ์
ตอบเป็น JSON เท่านั้น:
{{
    "analysis": "สรุปผลการวิเคราะห์ discrepancies และจุดอ่อนของ prompt",
    "new_prompt": "prompt ใหม่ทั้งหมดสำหรับ scorer"
}}
"""

SCORING_CRITERIA = {
    "reference": """1.0 - อ้างอิงมาตรากฎหมายถูกต้องและครอบคลุมตามเฉลย
0.0 - อ้างอิงมาตรากฎหมายไม่ถูกต้อง หรือไม่เกี่ยวข้อง
0.5 - อ้างอิงมาตรากฎหมายถูกต้อง แต่ไม่ครอบคลุม มีมาตราสำคัญตกหล่น""",
    "judgement": """1.0 - ตอบได้ถูกต้อง ชัดเจน ครอบคลุม
0.0 - ไม่ถูกต้องตามเฉลย
0.5 - ตอบได้ถูกต้อง แต่ไม่ครอบคลุม มีเนื้อหาสำคัญตกหล่น""",
    "suggestion": """1.0 - ตอบได้ถูกต้อง ชัดเจน ครอบคลุม ให้ข้อเสนอแนะที่เป็นไปได้จริง
0.0 - ไม่ถูกต้องตามเฉลย
0.5 - ตอบได้ถูกต้อง แต่ไม่ครอบคลุม มีเนื้อหาสำคัญตกหล่น""",
}

VERIFICATION_INSTRUCTIONS = {
    "reference": "ใช้ File Search ค้นหาตัวบทกฎหมายจริงจาก file store เพื่อตรวจสอบว่ามาตราที่อ้างอิงใน output ถูกต้องและครบถ้วนหรือไม่",
    "judgement": "ใช้ File Search ค้นหาคำพิพากษาศาลฎีกาจริงจาก file store เพื่อตรวจสอบว่าเลขฎีกาและเนื้อหาที่อ้างอิงใน output มีอยู่จริงและเกี่ยวข้องหรือไม่",
    "suggestion": "วิเคราะห์จากเนื้อหาใน review data โดยเน้นความเฉพาะเจาะจง (Specificity) และการนำไปใช้จริงในบริบทธนาคาร",
}

In [ ]:
def run_reviewer_agent(scorer_name, review_data, current_prompt, file_store_names=None):
    """Run a reviewer agent for a specific scorer section.
    
    Args:
        scorer_name: 'reference', 'judgement', or 'suggestion'
        review_data: list of dicts with scores, rationale, feedback
        current_prompt: current scorer prompt text
        file_store_names: list of file store IDs for File Search (None = no File Search)
    
    Returns:
        dict with 'analysis' and 'new_prompt' keys
    """
    instruction = AGENT_INSTRUCTION.format(
        scorer_section=scorer_name,
        scoring_criteria=SCORING_CRITERIA[scorer_name],
        verification_instruction=VERIFICATION_INSTRUCTIONS[scorer_name],
    )
    
    contents = f"""{instruction}

## Current Prompt
{current_prompt}

## Review Data
{json.dumps(review_data, ensure_ascii=False, indent=2)}
"""
    
    # Configure tools
    tools = None
    if file_store_names:
        tools = [
            types.Tool(
                file_search=types.FileSearch(
                    file_search_store_names=file_store_names,
                )
            )
        ]
    
    response = client.models.generate_content(
        model=REVIEW_AI_MODEL,
        contents=contents,
        config=types.GenerateContentConfig(
            temperature=0,
            tools=tools,
        ),
    )
    
    # Parse JSON from response
    text = response.text
    start_idx = text.find('{')
    end_idx = text.rfind('}') + 1
    if start_idx >= 0 and end_idx > start_idx:
        return json.loads(text[start_idx:end_idx])
    
    return {"analysis": text, "new_prompt": ""}

### 5.1 Reference Reviewer Agent
Access to general_law + specific_law File Search stores

In [ ]:
print("Running Reference Reviewer Agent...")
reference_result = run_reviewer_agent(
    scorer_name="reference",
    review_data=per_scorer_data["reference"],
    current_prompt=reference_prompt,
    file_store_names=[GENERAL_LAW_FILE_STORE, SPECIFIC_LAW_FILE_STORE],
)
print("Done.\n")
print("=== Analysis ===")
print(reference_result["analysis"])

### 5.2 Judgement Reviewer Agent
Access to supreme_court File Search store

In [ ]:
print("Running Judgement Reviewer Agent...")
judgement_result = run_reviewer_agent(
    scorer_name="judgement",
    review_data=per_scorer_data["judgement"],
    current_prompt=judgement_prompt,
    file_store_names=[SUPREME_COURT_STATEMENT_FILE_STORE],
)
print("Done.\n")
print("=== Analysis ===")
print(judgement_result["analysis"])

### 5.3 Suggestion Reviewer Agent
No File Search — review only

In [ ]:
print("Running Suggestion Reviewer Agent...")
suggestion_result = run_reviewer_agent(
    scorer_name="suggestion",
    review_data=per_scorer_data["suggestion"],
    current_prompt=suggestion_prompt,
    file_store_names=None,
)
print("Done.\n")
print("=== Analysis ===")
print(suggestion_result["analysis"])

## 6. Review New Prompts

In [ ]:
print("=" * 60)
print("REFERENCE - New Prompt")
print("=" * 60)
print(reference_result.get("new_prompt", "(empty)"))
print()
print("=" * 60)
print("JUDGEMENT - New Prompt")
print("=" * 60)
print(judgement_result.get("new_prompt", "(empty)"))
print()
print("=" * 60)
print("SUGGESTION - New Prompt")
print("=" * 60)
print(suggestion_result.get("new_prompt", "(empty)"))

## 7. Save New Prompts

In [ ]:
prompt_dir = os.path.join("./skill_archive", "eval_scorer", EVAL_AI_MODEL)

for scorer_name, result in [("reference", reference_result), ("judgement", judgement_result), ("suggestion", suggestion_result)]:
    new_prompt = result.get("new_prompt", "")
    if not new_prompt:
        print(f"WARNING: {scorer_name} has empty new_prompt, skipping")
        continue
    
    filename = f"legal_{scorer_name}_scorer_{NEW_EVAL_AI_PROMPT_VERSION}.md"
    path = os.path.join(prompt_dir, filename)
    with open(path, "w", encoding="utf-8") as f:
        f.write(new_prompt)
    print(f"Saved: {path}")